### Install libraries 

First, install the libraries to be able to load fine-tuned model

In [1]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("Device Name:", torch.cuda.get_device_name(0))

CUDA Available: True
Device Name: NVIDIA L4


In [2]:
import transformers
print(transformers.__version__)

/opt/conda/envs/gemma-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4.53.1


In [3]:
import sys
print(sys.executable)

/opt/conda/envs/gemma-env/bin/python


In [4]:
from transformers import pipeline

In [5]:
g3_model = "gs://mlops-course-dulcet-bastion-452612-v4-unique/week10/fine-tuning/output/gemma-3-1b-it-1751096651913-20250628013537/merged_model"
local_dir = "g3_plain_model"
model = g3_model

In [6]:
!rm -rf $local_dir
!mkdir -p $local_dir
!gsutil -m cp -r $model $local_dir

Copying gs://mlops-course-dulcet-bastion-452612-v4-unique/week10/fine-tuning/output/gemma-3-1b-it-1751096651913-20250628013537/merged_model/added_tokens.json...
Copying gs://mlops-course-dulcet-bastion-452612-v4-unique/week10/fine-tuning/output/gemma-3-1b-it-1751096651913-20250628013537/merged_model/config.json...
Copying gs://mlops-course-dulcet-bastion-452612-v4-unique/week10/fine-tuning/output/gemma-3-1b-it-1751096651913-20250628013537/merged_model/generation_config.json...
Copying gs://mlops-course-dulcet-bastion-452612-v4-unique/week10/fine-tuning/output/gemma-3-1b-it-1751096651913-20250628013537/merged_model/pytorch_model.bin...
Copying gs://mlops-course-dulcet-bastion-452612-v4-unique/week10/fine-tuning/output/gemma-3-1b-it-1751096651913-20250628013537/merged_model/tokenizer.json...
Copying gs://mlops-course-dulcet-bastion-452612-v4-unique/week10/fine-tuning/output/gemma-3-1b-it-1751096651913-20250628013537/merged_model/special_tokens_map.json...
Copying gs://mlops-course-dulcet

In [7]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

local_path=local_dir+"/merged_model/"
tokenizer = AutoTokenizer.from_pretrained(local_path)
model = AutoModelForCausalLM.from_pretrained(local_path)

In [8]:
def gemma3_prompt(text):
    prompt = (
    "<start_of_turn>system\n"
    "Classify the flower based on its measurements into one of the following species: [Setosa, Versicolor, Virginica]\n"
    "<end_of_turn>\n"
    "<start_of_turn>user\n"
    +text+
    "<end_of_turn>\n"
    "<start_of_turn>assistant\n"
    )
    
    return prompt

In [16]:
text= (
    #"Sepal Length: 6.4, Sepal Width: 2.9, Petal Length: 4.3, Petal Width: 1.3" #versicolor
    "Sepal Length: 5.0, Sepal Width: 3.6, Petal Length: 1.4, Petal Width: 0.2" #setosa
)

In [17]:
prompt = gemma3_prompt(text)
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(
    **inputs,
    max_new_tokens=2,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

# Decode only the new tokens
generated_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
print("Predicted class:", generated_text.strip())

Predicted class: Versicolor
